# Ovarian Cancer Detection: Preprocessing & Leakage Check
Aaishah Khan

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [ ]:
df = pd.read_csv("oc_risk50k_single_dataset.csv")
print(df.shape)

In [ ]:
# drop ID + post-diagnosis columns (leakage)
cols_removed = ["Patient_ID", "Risk_Category", "Follow_Up_Status",
                "Treatment_Status", "Time_To_Progression"]
df = df.drop(columns=cols_removed)

target = "Ovarian_Cancer_Status"
X = df.drop(columns=[target, "Partition"])
y = df[target]

x_train, y_train = X[df.Partition == "Train"], y[df.Partition == "Train"]
x_test,  y_test  = X[df.Partition == "Validation"], y[df.Partition == "Validation"]
print(x_train.shape, x_test.shape)

## 1. Missing values

In [ ]:
# missing values (train set)
missing = x_train.isnull().sum()
missing_pct = (missing[missing > 0] / len(x_train) * 100).sort_values()

ax = missing_pct.plot(kind="barh", color="#7A2E1F", figsize=(8, 3))
ax.set_xlabel("% of training rows missing")
ax.set_title("Missing values by feature (train set)")
plt.tight_layout()
plt.savefig("MissingValues.png", dpi=200)
plt.show()

Only 4 features have missing values (~4% each). Dropping incomplete rows would lose ~15% of the data, so imputing instead.

## 2. Median imputation + scaling

In [ ]:
# median imputation (outliers kept) + scaling
def evaluate(X_tr, X_te):
    model = make_pipeline(SimpleImputer(strategy="median"),
                          StandardScaler(),
                          LogisticRegression(max_iter=1000))
    model.fit(X_tr, y_train)
    pred = model.predict(X_te)
    return {"Accuracy": round(accuracy_score(y_test, pred), 3),
            "Precision": round(precision_score(y_test, pred), 3),
            "Recall": round(recall_score(y_test, pred), 3),
            "F1": round(f1_score(y_test, pred), 3)}

print("All features:   ", evaluate(x_train, x_test))

## 3. Leakage check: drop imaging/tumor features

In [ ]:
# imaging/tumor features
imaging_cols = ["Tumor_Size", "Laterality", "Tumor_Morphology", "Solid_Component",
                "Ascites_Status", "Bilateral_Mass", "Papillary_Projection",
                "Doppler_Vascularity", "Lymph_Node_Enlargement", "Imaging_Risk_Score"]

print("Without imaging:", evaluate(x_train.drop(columns=imaging_cols),
                                   x_test.drop(columns=imaging_cols)))

## 4. Why is it still perfect? Single-feature check

In [ ]:
# ROC-AUC per feature
single_auc = {}
for col in x_test.columns:
    auc = roc_auc_score(y_test, x_test[col].fillna(x_train[col].median()))
    single_auc[col] = max(auc, 1 - auc)

pd.Series(single_auc).sort_values(ascending=False).head(10).round(3)

## Takeaways
- Missing data is small (~4% in 4 features), handled with median imputation
- Still 100% without the imaging/tumor features, so imaging leakage isn't the cause
- Single biomarkers (CA-125, LDH, HE4) separate the classes almost perfectly on their own, so the synthetic data is too clean